# Session 1 — Working with real social-science data in R

**Course:** R for Research in the Humanities and Social Sciences - CUSO 
**Format:** online, 2 hours  
**Dataset:** `carData::SLID` — Survey of Labour and Income Dynamics  

## What we will do

This first session is built around a real social-science dataset rather than a toy example. `SLID` contains observations from the 1994 wave of the Canadian Survey of Labour and Income Dynamics in Ontario. Each row is a respondent; the variables include hourly wages, years of education, age, sex, and language. There are also missing values, especially for wages.

That makes it useful for learning R because the structure resembles many research datasets in the social sciences, psychology, education, and related fields:

> **individuals → measurements → groups → missing data → summaries → figures**

The goal today is not to learn every feature of R. The goal is to understand what R is doing, line by line, and to acquire a small vocabulary that already lets us answer a real empirical question.

Our workflow will be:

> **Question → inspect → clean → transform → summarise → visualise → interpret**

The main empirical question will be:

> **How is education associated with hourly wages in this sample?**

We will also ask whether the pattern looks similar across groups. We will remain descriptive today: association is not causation.

---

# Before the session

This notebook is designed to run in **Google Colab with an R runtime**. You do not need to install R or RStudio on your own computer.

In a fresh Colab session, run the installation cell below first. Colab sessions are temporary, so you may need to install the packages again the next time you open a fresh runtime.

In [ ]:
# install.packages() downloads and installs an R package.
# The package name is written between quotation marks because it is text.
#
# We need two packages for this session:
#   - tidyverse: tools for manipulating and visualising data;
#   - carData: the package that contains the SLID teaching dataset.
#
# We install them one at a time so that the syntax is easy to read.
# In Colab, packages may need to be installed again in a new runtime.

install.packages("tidyverse")
install.packages("carData")

Installing and loading are two different operations:

In [ ]:
# install.packages("tidyverse") installs a package into the current environment.
#
# library(tidyverse) loads that installed package so that its functions
# are available in the current R session.
#
# We will load tidyverse again below when we begin working with the dataset.
library(tidyverse)

When you later work with your own files, avoid paths that exist only on one specific computer. Relative paths make code easier to reuse.

In [ ]:
# Avoid paths such as:
# setwd("C:/Users/myname/Desktop/project")
#
# Prefer project-relative paths such as:
# "data/my_file.csv"
# "figures/my_plot.png"

---

# 1. R is a calculator, but one that remembers things
We begin in the Console or in an R script.

Anything following `#` is a **comment**. R ignores it. Comments are for humans — including your future self.

In [ ]:
# R can evaluate arithmetic expressions directly.

# Addition
1 + 1

# Multiplication
3 * 4

# Division
10 / 4

# Powers: 2^3 means 2 × 2 × 2
2^3

# Parentheses control the order of operations.
# R calculates (2 + 3) first, then multiplies the result by 4.
(2 + 3) * 4

When you run one of these lines, R:

1. reads the expression;
2. evaluates it;
3. prints the result.

## Numbers and text are different

In [ ]:
# The value below is numeric: R can use it in arithmetic.
12

# Quotation marks turn the same characters into text.
# R now treats "12" as a character string, not as the number twelve.
"12"

# Numeric values can be added.
12 + 1

Now deliberately create an error:

In [ ]:
# Run this line deliberately.
# It produces an error because "12" is text and 1 is numeric.
# The point is to see that an error message is information, not a disaster.
"12" + 1

Errors are normal. An error message tells us where R could not understand or execute an instruction.

## Store a value in an object

In [ ]:
# <- is the assignment operator.
# Read the next line as: "store the value 10 in an object called x".
x <- 10

# Typing the object's name asks R to display the value currently stored in it.
x

# Once an object exists, we can reuse it in another expression.
x + 5

# Assignment can also replace the previous value.
# After this line, x no longer contains 10: it contains 100.
x <- 100
x

Object names should tell us what they contain.

In [ ]:
# Short names are legal, but they quickly become hard to understand.
x <- 14

# Descriptive names make the meaning of the object visible in the code.
# Underscores are commonly used to separate words in R object names.
mean_years_education <- 14

# Display the value we just stored.
mean_years_education

## Functions

A function takes input, performs an operation, and returns output.

In [ ]:
# A function is called by writing its name followed by parentheses.

# sqrt() takes a number and returns its square root.
sqrt(16)

# round() can take several named arguments.
# Here x is the number to round and digits says how many decimals to keep.
round(x = 3.1415926, digits = 2)

The general shape is:

```text
function(argument_1, argument_2, ...)
```

You do not need to memorise every function. You need to learn how to read a function call.

In [ ]:
# A question mark followed by a function name opens its help page in RStudio.
# Documentation is part of normal R work: you are not expected to memorise everything.
?round

---

# 2. Load the tools and the dataset
## Load the tidyverse

In [ ]:
# library() loads an installed package into the current R session.
#
# tidyverse is a collection of packages. In this session we mainly use:
#   - dplyr for data manipulation;
#   - ggplot2 for visualisation.
library(tidyverse)

You may see messages about packages being attached or functions being masked. That is normal.

## Load the SLID dataset

The dataset is distributed with the `carData` R package.

In [ ]:
# data() loads a dataset that is distributed inside an R package.
#
# "SLID" is the name of the dataset.
# package = "carData" tells R where to find it.
data("SLID", package = "carData")

We now have an object called `SLID`.

In [ ]:
# head() displays the first six rows by default.
# This is a quick way to check what the dataset actually looks like.
head(SLID)

For the rest of the session, we will use a lower-case object name. This does not change the data; it simply creates another reference to the same values.

In [ ]:
# as_tibble() converts the ordinary data frame into a tidyverse "tibble".
# The values do not change; a tibble simply prints a compact preview,
# which is much easier to read in the console or notebook.
slid <- as_tibble(SLID)

# Typing the object's name now displays a manageable preview.
slid

## What does one row mean?

This question should always come before statistical analysis.

In [ ]:
# nrow() asks: how many observations (rows) are there?
nrow(slid)

# ncol() asks: how many variables (columns) are there?
ncol(slid)

# dim() returns both dimensions at once:
# first the number of rows, then the number of columns.
dim(slid)

Here, **one row corresponds to one respondent** in this extract of the survey.

That means that when we calculate something such as a mean age, the observational unit is the respondent — not a province, a household, or a year.

---

# 3. Never analyse a dataset you have not inspected
## Look at names

In [ ]:
# names() returns the names of all columns in a data frame.
# Before analysing data, check that the variables you expect are actually present.
names(slid)

The variables are:

```text
wages       hourly wage rate
education   years of schooling
age         age in years
sex         recorded sex category
language    language category
```

## Look at the first rows

In [ ]:
# head(slid) shows the first six observations.
head(slid)

# A second argument lets us choose how many rows to display.
# Here we ask for the first ten.
head(slid, 10)

## Look at structure and data types

In [ ]:
# glimpse() gives a compact overview of the whole data frame.
# For each variable, look at:
#   - its name;
#   - its type (<dbl>, <fct>, ...);
#   - a few example values.
glimpse(slid)

You should see types such as:

```text
<dbl>  numeric value, usually allowing decimals
<int>  integer
<fct>  factor: a categorical variable with defined levels
<chr>  character/text
```

Data types matter. Taking the mean of `age` makes sense. Taking the mean of `language` does not.

Check individual variables:

In [ ]:
# The $ operator extracts one column from a data frame.
# class() tells us how R is storing that column.

class(slid$wages)     # numeric wage values
class(slid$age)       # numeric age values
class(slid$sex)       # categorical variable
class(slid$language)  # categorical variable

## A quick summary

In [ ]:
# summary() gives a quick variable-by-variable overview.
#
# For numeric variables it reports quantities such as the minimum,
# quartiles, median, mean and maximum.
# For categorical variables it reports counts by category.
# It also reports missing values when they are present.
summary(slid)

Do not rush past this output. Ask:

- Are values in plausible ranges?
- Are there missing observations?
- Which variables are continuous?
- Which are categorical?
- Does the coding match what we think the variable means?

This habit catches many errors before they become results.

---

# 4. Columns are vectors — and missing values are values too
A data-frame column can be extracted with `$`.

In [ ]:
# The $ operator extracts one column from a data frame.
# The result is a vector: one age value for each respondent.
#
# We use head(..., 10) so that we see only the first ten values,
# rather than printing thousands of ages.
head(slid$age, 10)

This is a **vector**: an ordered collection of values.

In [ ]:
# Store the age column in a separate object called age.
age <- slid$age

# head(..., 10) displays only the first ten values.
# This is easier to inspect than printing the entire vector.
head(age, 10)

# length() counts how many values the vector contains.
# Here that should match the number of rows in slid.
length(age)

We can apply functions to a vector.

In [ ]:
# Functions can summarise all values in a numeric vector.

# Arithmetic mean
mean(age)

# Median: half the observations are below it and half are above it.
median(age)

# Standard deviation: a common measure of dispersion around the mean.
sd(age)

Now try wages:

In [ ]:
# Try the same operation on wages.
# R returns NA rather than a number because at least one wage value is missing.
mean(slid$wages)

Why?

Because the wage variable contains missing values.

In [ ]:
# is.na() checks every value and returns TRUE when it is missing
# and FALSE when it is observed.
#
# Show only the first 20 results so that the output stays readable.
head(is.na(slid$wages), 20)

# In R, TRUE behaves like 1 and FALSE like 0 in a sum.
# Summing the logical vector therefore counts the missing wage values.
sum(is.na(slid$wages))

# Taking the mean of TRUE/FALSE values gives a proportion.
# This is the share of wage observations that are missing.
mean(is.na(slid$wages))

To compute a mean while excluding missing observations from **that calculation**:

In [ ]:
# na.rm = TRUE tells mean() to remove missing values for this calculation.
# It does NOT modify the dataset itself.
mean(slid$wages, na.rm = TRUE)

Read `na.rm = TRUE` as:

> "remove missing values before applying this function."

Other examples:

In [ ]:
# The same na.rm argument can be used with many summary functions.

# Median of the observed wages
median(slid$wages, na.rm = TRUE)

# Standard deviation of the observed wages
sd(slid$wages, na.rm = TRUE)

# Smallest observed wage
min(slid$wages, na.rm = TRUE)

# Largest observed wage
max(slid$wages, na.rm = TRUE)

Important research point: `na.rm = TRUE` is a computational instruction, **not a missing-data strategy**. In a real study we would also ask why wages are missing and whether missingness is related to other variables.

---

# 5. Manipulate observations with `dplyr`
Most research questions require us to keep, transform, or combine variables and observations.

The `dplyr` package, loaded with the tidyverse, uses verbs that correspond closely to those operations.

## `filter()` keeps rows

In [ ]:
# filter() keeps ROWS that satisfy a condition.
#
# The first argument is the data frame.
# age >= 30 is the condition: keep a row when it is TRUE.
filter(slid, age >= 30)

A condition produces TRUE/FALSE values.

In [ ]:
# Before using filter(), it helps to see what the condition itself produces.
# R evaluates age >= 30 once for every respondent.
# The result is a vector of TRUE and FALSE values.
#
# Again, show only the first 20 values to keep the output readable.
head(slid$age >= 30, 20)

`filter()` retains the rows for which the condition is TRUE.

Several conditions can be combined.

In [ ]:
# Several conditions separated by commas inside filter() are combined with AND.
# A row is kept only if ALL three conditions are true.

filter(
  slid,             # start from the slid data frame
  age >= 30,        # age is at least 30
  age <= 40,        # age is at most 40
  !is.na(wages)     # ! means NOT: wage is not missing
)

Useful operators:

```text
==   equal to
!=   not equal to
>    greater than
<    less than
>=   greater than or equal to
<=   less than or equal to
!    logical NOT
&    AND
|    OR
```

For a categorical variable:

In [ ]:
# For categorical variables we usually test equality with ==.
# This keeps only rows whose language category is exactly "French".
#
# Be careful: = and == have different roles in R.
filter(slid, language == "French")

## `select()` keeps columns

In [ ]:
# select() keeps COLUMNS rather than rows.
# Here we create a temporary view containing only four variables.
select(
  slid,
  wages,
  education,
  age,
  language
)

## `arrange()` sorts rows

In [ ]:
# arrange() changes the order of the rows.

# Sort from the smallest wage to the largest.
arrange(slid, wages)

# desc() reverses the order:
# sort from the largest wage to the smallest.
arrange(slid, desc(wages))

Notice that none of these commands modifies `slid`. They return a new result.

---

# Short break
Take a short break away from the screen.

---

# 6. The pipe: write code in the order you think
Suppose we want to:

1. start with `slid`;
2. keep respondents with an observed wage;
3. keep only a few variables;
4. sort from highest to lowest wage.

We could nest functions, but it quickly becomes difficult to read.

The native R pipe `|>` means approximately **"then"**.

In [ ]:
# The native R pipe |> passes the result of one step into the next step.
# Read it aloud as "then".

slid |>                              # start with the full dataset
  filter(!is.na(wages)) |>           # THEN keep rows with an observed wage
  select(wages, education, age, language) |>  # THEN keep four columns
  arrange(desc(wages))               # THEN sort wages from high to low

Read the code aloud:

> Take `slid`, then filter, then select, then arrange.

The following two pieces of code are equivalent:

In [ ]:
# Without a pipe, the data frame is written explicitly as the first argument.
filter(slid, age >= 30)

In [ ]:
# With a pipe, slid is passed automatically into filter().
# This produces exactly the same result as the previous command.
slid |>
  filter(age >= 30)

The pipe becomes useful when the analysis has several consecutive steps.

## Save the result

In [ ]:
# A pipeline returns a result just like any other R expression.
# Using <- lets us save that result in a new object.

slid_complete_wage <- slid |>
  filter(!is.na(wages))

# The original dataset has not changed.
# Compare its number of rows with the new filtered dataset.
nrow(slid)
nrow(slid_complete_wage)

We have **not** overwritten the original data. Keeping the raw object unchanged is usually a good habit.

## `mutate()` creates variables

A common task in research is deriving a variable from existing measurements.

For example, create broad age groups:

In [ ]:
# mutate() adds or transforms columns while keeping the rows.
#
# We create a new variable called age_group from the numeric age variable.

slid_with_age_group <- slid |>
  mutate(
    age_group = case_when(
      # case_when() tests conditions from top to bottom.
      # The first matching condition determines the label.
      age < 30 ~ "under 30",
      age < 45 ~ "30–44",
      age < 60 ~ "45–59",
      age >= 60 ~ "60+"
    )
  )

# Inspect the result by showing only age and the newly created age_group.
# head(20) limits the output to the first 20 rows.
slid_with_age_group |>
  select(age, age_group) |>
  head(20)

The original `slid` object still has only the original variables:

In [ ]:
# names(slid) shows that the original object is unchanged:
# it still contains only the original variables.
names(slid)

The modified object contains the new one:

In [ ]:
# The new object contains one additional variable: age_group.
names(slid_with_age_group)

### Mini exercise

Create an object called `working_sample` that:

- keeps only respondents with observed wages;
- keeps only `wages`, `education`, `age`, `sex`, and `language`;
- sorts rows from highest to lowest wage.

Try it before opening the solution.

<details>
<summary>Solution</summary>

In [ ]:
# Start from the original data.
working_sample <- slid |>
  # Keep only respondents for whom wage is observed.
  filter(!is.na(wages)) |>
  # Keep the five variables relevant to the exercise.
  select(wages, education, age, sex, language) |>
  # Sort the resulting rows from highest to lowest wage.
  arrange(desc(wages))

# Display the object we have just created.
working_sample

</details>

---

# 7. Go from individual observations to evidence about groups
## Count observations

In [ ]:
# count(variable) counts how many rows belong to each category.
#
# The output has:
#   - one row per language category;
#   - a column called n containing the count.
slid |>
  count(language)

`count()` is useful whenever you need to understand the composition of a dataset.

We can immediately turn counts into percentages of **this sample**:

In [ ]:
# First count the respondents in each language category.
slid |>
  count(language) |>
  # count() created a column called n.
  # sum(n) is the total sample size.
  # Dividing each group count by the total gives its proportion;
  # multiplying by 100 expresses that proportion as a percentage.
  mutate(
    percent = 100 * n / sum(n)
  )

Be precise in interpretation: these are proportions in the dataset we are analysing. Survey weights and sampling design would matter before treating them as population estimates.

## One summary for the entire dataset

In [ ]:
# summarise() reduces many rows to a smaller set of summary statistics.
# Without group_by(), the calculations use the whole dataset.

slid |>
  summarise(
    # Mean age across all respondents with an observed age
    mean_age = mean(age, na.rm = TRUE),

    # Mean wage across respondents with an observed wage
    mean_wage = mean(wages, na.rm = TRUE)
  )

`summarise()` reduces many rows to one or more summary values.

## Summaries by group

Now ask:

> What do observed wages look like in each language category?

In [ ]:
# group_by(language) tells R to treat each language category separately.
# summarise() is then calculated once inside each group.

slid |>
  group_by(language) |>
  summarise(
    # n() counts all rows in the current group.
    n = n(),

    # Count only respondents whose wage is not missing.
    n_wage = sum(!is.na(wages)),

    # Mean wage among observed wage values.
    mean_wage = mean(wages, na.rm = TRUE),

    # Median wage among observed wage values.
    # The median is often useful when a distribution is skewed.
    median_wage = median(wages, na.rm = TRUE),

    # Standard deviation describes dispersion around the mean.
    sd_wage = sd(wages, na.rm = TRUE)
  )

The logic is important:

```text
group_by(language)
        ↓
English observations  → summarise()
French observations   → summarise()
Other observations    → summarise()
        ↓
one row per group
```

Now look at education:

In [ ]:
# The same group_by() + summarise() logic can be reused
# with a different outcome variable.

slid |>
  group_by(language) |>
  summarise(
    # Number of respondents in the language group
    n = n(),

    # Average years of education in that group
    mean_education = mean(education, na.rm = TRUE),

    # Dispersion of years of education in that group
    sd_education = sd(education, na.rm = TRUE)
  )

A crucial lesson: a difference between group means is a **description**. It does not by itself tell us why the groups differ.

---

# 8. Make the observations visible with `ggplot2`
`ggplot2` builds figures from layers. Instead of memorising complete plotting commands, we will construct a figure step by step.

The three pieces to remember are:

```text
data        which data frame?
aesthetics  which variables become x, y, colour, etc.?
geometry    points, bars, lines, boxes, ...?
```

## Step 1 — declare the data

In [ ]:
# ggplot() starts a plot.
# At this stage we give it only the dataset.
# Because we have not yet defined axes or marks, the plot is empty.
ggplot(data = slid)

Nothing is visible yet because we have not said what to plot.

## Step 2 — map a variable to an axis

In [ ]:
# aes() defines an aesthetic mapping:
# it tells ggplot which data variable should control a visual property.
#
# Here the wages variable is mapped to the x-axis.
# We still have not chosen how the observations should be drawn.
ggplot(
  data = slid,
  aes(x = wages)
)

Still no marks: we have defined an axis but not a geometry.

## Step 3 — add a geometry

In [ ]:
# geom_histogram() chooses the geometry:
# it divides a numeric variable into bins and counts observations in each bin.
#
# ggplot(...) defines the data and mapping.
# The + sign adds another layer to the plot.
ggplot(data = slid, aes(x = wages)) +
  geom_histogram()

You may see a warning about removed rows containing missing values. That warning is useful: `ggplot2` is telling us that some wage observations could not be drawn.

We can make the analytic choice explicit:

In [ ]:
# We can remove missing wage values explicitly before plotting.
# This makes the data-selection step visible in the code.

slid |>
  filter(!is.na(wages)) |>       # keep only observed wages
  ggplot(aes(x = wages)) +       # map wage to the x-axis
  geom_histogram()               # draw a histogram

## Relationship between two numerical variables

Our main question concerns education and wages.

In [ ]:
# A scatterplot represents the relationship between two numeric variables.
#
# education is mapped to x, wages to y.
# Each point corresponds to one respondent with usable values.
ggplot(
  slid,
  aes(x = education, y = wages)
) +
  geom_point()

With thousands of observations, points overlap. We can make them partly transparent.

In [ ]:
# With many respondents, points overlap.
# alpha controls transparency:
#   1    = fully opaque
#   0.25 = mostly transparent
#
# Because alpha is a fixed visual setting rather than a data variable,
# it is written outside aes().
ggplot(slid, aes(x = education, y = wages)) +
  geom_point(alpha = 0.25)

`alpha = 0.25` is **not** mapped to a variable. It is a fixed visual setting, so it is written outside `aes()`.

Compare:

In [ ]:
# FIXED visual property:
# every point receives exactly the same transparency.
#
# alpha is outside aes() because it is not linked to a variable.
ggplot(slid, aes(x = education, y = wages)) +
  geom_point(alpha = 0.25)

with:

In [ ]:
# MAPPED visual property:
# colour is inside aes() because colour now represents the variable language.
#
# ggplot automatically creates a legend for mapped aesthetics.
ggplot(
  slid,
  aes(
    x = education,
    y = wages,
    colour = language
  )
) +
  geom_point(alpha = 0.25)

This distinction — **inside vs outside `aes()`** — is one of the most important things to understand in `ggplot2`.

## Add a descriptive trend

In [ ]:
# Start with the scatterplot.
ggplot(slid, aes(x = education, y = wages)) +
  # Draw the individual respondents with transparent points.
  geom_point(alpha = 0.20) +
  # Add a straight fitted line.
  # method = "lm" asks for a linear model.
  # We use it here only as a descriptive summary of the association;
  # the statistical model itself will be studied in Session 2.
  geom_smooth(method = "lm")

At this stage the line is a visual summary of association. Do not interpret it as evidence that education itself *causes* a particular wage increase.

## Compare groups without putting everything on one plot

Facets create one panel per category.

In [ ]:
# facet_wrap() makes separate panels for categories of a variable.
# The same x and y mapping is reused in every panel.

ggplot(slid, aes(x = education, y = wages)) +
  geom_point(alpha = 0.20) +
  geom_smooth(method = "lm") +
  # ~ language means: create one panel for each language category.
  facet_wrap(~ language)

The formula:

In [ ]:
# In formulas used by functions such as facet_wrap(),
# the tilde ~ can be read roughly as "according to".
#
# Here: make panels according to language.
~ language

means: create panels according to values of `language`.

## Add human-readable labels

In [ ]:
# A plot can be built step by step and stored in an object.
wage_plot <- slid |>
  # Be explicit that only observed wages are used in the plot.
  filter(!is.na(wages)) |>
  # Define the variables on the x and y axes.
  ggplot(aes(x = education, y = wages)) +
  # Draw individual respondents.
  geom_point(alpha = 0.20) +
  # Add a descriptive linear trend.
  geom_smooth(method = "lm") +
  # Put language categories in separate panels.
  facet_wrap(~ language) +
  # labs() replaces technical variable names with labels for readers.
  labs(
    title = "Education and hourly wages in the SLID sample",
    subtitle = "Ontario, 1994; panels show recorded language category",
    x = "Years of education",
    y = "Hourly wage",
    caption = "Source: Survey of Labour and Income Dynamics (SLID)"
  ) +
  # theme_minimal() changes the overall visual appearance.
  theme_minimal()

# Display the finished plot.
wage_plot

Notice that a plot can itself be stored in an object:

In [ ]:
# Plots are R objects too.
# class() shows the class that R uses for the stored plot object.
class(wage_plot)

## Save a publication-quality image

Research code should be able to reproduce output files as well as results on screen.

In [ ]:
# dir.create() creates a folder.
# showWarnings = FALSE prevents a warning if the folder already exists.
dir.create("figures", showWarnings = FALSE)

# ggsave() writes a ggplot object to a file.
# This is reproducible: running the code again recreates the same output.
ggsave(
  filename = "figures/education_and_wages.png",  # output path
  plot = wage_plot,                              # plot object to save
  width = 8,                                     # width in inches
  height = 5,                                    # height in inches
  dpi = 300                                      # image resolution
)

This is much more reproducible than manually exporting the figure from RStudio.

---

# 9. Final challenge
Choose one question. If time is short, begin it now and finish after class.

### A — Education and wages

> How does hourly wage vary with years of education in this sample?

### B — Age and wages

> What does the relationship between age and observed wages look like?

### C — Group composition

> How are respondents distributed across language categories, and how do their observed wage distributions differ?

For your chosen question, produce:

1. one filtered or transformed dataset if necessary;
2. one summary table;
3. one figure;
4. one sentence describing what the data show.

A possible skeleton:

In [ ]:
# STEP 1 — choose the observations needed for the analysis.
analysis_data <- slid |>
  filter(!is.na(wages))

# STEP 2 — turn individual observations into a summary table.
summary_table <- analysis_data |>
  group_by(language) |>
  summarise(
    n = n(),
    mean_wage = mean(wages),
    median_wage = median(wages)
  )

# Display the summary table.
summary_table

# STEP 3 — make the relationship visible.
analysis_plot <- analysis_data |>
  ggplot(aes(x = education, y = wages)) +
  geom_point(alpha = 0.20) +
  geom_smooth(method = "lm") +
  theme_minimal()

# Display the plot.
analysis_plot

A careful descriptive sentence would look like:

> "In this sample, respondents with more years of education tend to report higher observed hourly wages."

That is different from:

> "Additional education causes higher wages."

The second statement requires a causal research design that we have not established.

---

# 10. What you now know

The amount of syntax introduced today is deliberately small.

```text
# Inspect
head()
glimpse()
summary()
nrow()
names()

# Missingness
is.na()

# Manipulate
filter()
select()
arrange()
mutate()

# Group and summarise
count()
group_by()
summarise()

# Visualise
ggplot()
geom_histogram()
geom_point()
geom_smooth()
facet_wrap()
labs()

# Save output
ggsave()
```

The more important idea is the workflow:

```text
What is one row?
      ↓
What are the variables and their types?
      ↓
What is missing?
      ↓
Which observations do I need?
      ↓
What numerical summary answers my question?
      ↓
What figure lets me see the observations?
      ↓
What can I legitimately conclude?
```

In Session 2 we will move from a visible association to **statistical evidence**: estimates, uncertainty, confidence intervals, and regression models.

---

# Optional: how this translates to your own CSV file

Today we used a dataset bundled with an R package so that everybody could run exactly the same code. Your own research data will often arrive as CSV or Excel files.

For a CSV file, the workflow changes only at the import step:

In [ ]:
# read_csv() imports a comma-separated text file as a data frame.
# The path is relative to the project folder.
#
# This line is an example: it will work only if that file actually exists.
my_data <- read_csv("data/my_data.csv")

# Always inspect imported data before analysing it.
glimpse(my_data)
summary(my_data)

For an Excel file:

In [ ]:
# Excel support comes from the readxl package.
# Install it once on your computer if necessary:
# install.packages("readxl")

# Load the package into the current R session.
library(readxl)

# read_excel() imports an Excel workbook.
# Again, this is an example path: replace it with the path to your own file.
my_data <- read_excel("data/my_data.xlsx")

Everything that follows — `filter()`, `mutate()`, `group_by()`, `ggplot()` — works with the same logic.





## Dataset source

`SLID` is distributed in the `carData` package and is described as data from the 1994 wave of the Canadian Survey of Labour and Income Dynamics for Ontario, prepared from the public-use dataset made available by Statistics Canada.